In [0]:
# Nomes do catalog e das camadas
catalog = "cinedata"

bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

landing_path = f"/Volumes/{catalog}/bronze/landing"

In [0]:
from pyspark.sql import functions as F, Window

In [0]:
# info_filmes
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

df_silver_info = df_bronze_info

# remove duplicação, mantendo o mais recente
w = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_silver_info = (
    df_silver_info
    .withColumn("row_number", F.row_number().over(w))
    .filter(F.col("row_number") == 1)
)


# Converte a data
# formatos possíveis de data
formatos = [
    "yyyy-MM-dd",
    "yyyy/MM/dd",
    "dd/MM/yyyy",
    "MM/dd/yyyy",
    "dd-MM-yyyy",
    "MM-dd-yyyy",
]
# Faz o trim na data
data_limpa = F.trim(F.col("release_date"))
# Tenta converter para algum dos formatos de data
# O coalesce junta o resultado: se alguma conversão deu certo, é o valor que fica na coluna, se não fica null
df_silver_info = (
    df_silver_info
    .withColumn("release_date", F.coalesce(*[F.try_to_date(data_limpa, f) for f in formatos]))
)


# Para ver os status: display(df_bronze_info.groupBy("status").count())
# Baseado nos status vistos, normaliza os status
status_normalizado = F.lower(F.trim(F.col("status")))
# Troca hífens por espaço
status_normalizado = F.regexp_replace(status_normalizado, r"[-_.]+", " ")
# Traduz o status
df_silver_info = (
    df_silver_info
    .withColumn("status_normalizado", status_normalizado)
    .withColumn("status_filme", 
        F.when(F.col("status_normalizado") == "released", "Lançado")
        .when(F.col("status_normalizado") == "post production", "Pós produção") 
        .when(F.col("status_normalizado") == "in production", "Em Produção")
        .when(F.col("status_normalizado") == "planned", "Planejado")
        .when(F.col("status_normalizado") == "rumored", "Rumores")
        .when(F.col("status_normalizado") == "canceled", "Cancelado")
        .otherwise("Não Informado")
    )
)


# Renomeia colunas
df_silver_info = (
    df_silver_info
    .select("id", "title", "original_title", "release_date", "runtime", "original_language", "overview", "tagline", "status_filme")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
)

df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

In [0]:
# Cotação do dólar
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

# Converte os tipos
df_silver_cotacao = (
    df_bronze_cotacao
    .withColumn("data_hora_cotacao", F.try_to_timestamp(F.col("dataHoraCotacao")))
    .withColumn("data_cotacao", F.to_date("data_hora_cotacao"))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("decimal(10,5)"))
    .filter(
        F.col("data_cotacao").isNotNull()
        & F.col("cotacao_compra").isNotNull()
        & (F.col("cotacao_compra") > 0)
    )
)

# Deduplicação: mantém a última cotação do dia e, em caso de empate, a com a ingestão mais recente
w_dia = Window.partitionBy("data_cotacao").orderBy(F.col("data_hora_cotacao").desc(), F.col("ingestion_datetime").desc())
df_silver_cotacao = (
    df_silver_cotacao
    .withColumn("row_number", F.row_number().over(w_dia))
    .filter(F.col("row_number") == 1)
    .select("data_cotacao", "cotacao_compra")
)

# Limites: pega o rimeiro e último dias da série com cotação registrada
limites = df_silver_cotacao.agg(
    F.min("data_cotacao").alias("data_min"),
    F.max("data_cotacao").alias("data_max"),
).first()

# Gera todos os dias entre data_min e data_max, incluindo os feriados e fins de semana
df_todos_dias = spark.range(1).select(
    F.explode(
        F.sequence(F.lit(limites["data_min"]), F.lit(limites["data_max"]), F.expr("interval 1 day"))
    ).alias("data_cotacao")
)

# Para preencher os fins de semana e feriados com a cotação anterior:
# faz um lef join e usa uma janela para pegar o último valor não nulo antes da data
w_fill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_silver_cotacao = (
    df_todos_dias
    .join(df_silver_cotacao, on="data_cotacao", how="left")
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(w_fill)
    )
    .orderBy("data_cotacao")
)

df_silver_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# financeiro_filmes
df_bronze_financeiro = spark.table(f"{bronze_schema}.tb_movies_financials")

valores_ausentes = ["unknown", "não informado", "nao informado"]

df_silver_financeiro = (
    # Textos se tornam null
    df_bronze_financeiro
    .withColumn("orcamento", 
        F.when(F.lower(F.trim(F.col("budget"))).isin(valores_ausentes), F.lit(None))
        .otherwise(F.lower(F.trim(F.col("budget"))))
    )
    .withColumn("receita", 
        F.when(F.lower(F.trim(F.col("revenue"))).isin(valores_ausentes), F.lit(None))
        .otherwise(F.lower(F.trim(F.col("revenue"))))
    )
)

# Tira símbolos
def tirar_simbolos(coluna):
    # Tira tudo que não é número, ponto, vírgula e sinal de menos
    limpo = F.regexp_replace(F.col(coluna), r"[^0-9.,\-]", "")
    # Remove separador de mil em mil
    limpo = F.regexp_replace(limpo, r"[.,](?=\d{3}(?:[.,]|$))", "")
    # Substitui a vírgula por ponto
    return F.regexp_replace(limpo, ",", ".")

df_silver_financeiro = (
    df_silver_financeiro
    .withColumn("orcamento", tirar_simbolos("orcamento"))
    .withColumn("receita", tirar_simbolos("receita"))
)

# Converte de string para números decimais
def converte_decimal(coluna):
    valor = coluna.try_cast("decimal(18, 2)")
    return F.when(valor > 0, valor).otherwise(F.lit(None))

df_silver_financeiro = (
    df_silver_financeiro
    .withColumn("orcamento_usd", converte_decimal(F.col("orcamento")))
    .withColumn("receita_usd", converte_decimal(F.col("receita")))
)

# Seleciona a cotação para converter de dólares para reais
taxa = (
    spark.table(f"{silver_schema}.tb_cotacao_dolar")
    .orderBy(F.col("data_cotacao").desc())
    .select("cotacao_compra")
    .first()
)
taxa_dolar = float(taxa[0])

df_silver_financeiro = (
    df_silver_financeiro
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * taxa_dolar, 2))
    .withColumn("receita_brl",   F.round(F.col("receita_usd")   * taxa_dolar, 2))
)

# Calcula o lucro (receita - orcamento) e margem de lucro (lucro / receita * 100)
df_silver_financeiro = (
    df_silver_financeiro
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")))
    .withColumn(
        "margem_lucro_pct",
        F.when(
            F.col("receita_usd").isNotNull() & (F.col("receita_usd") != 0) & F.col("lucro_usd").isNotNull(),
            F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2)
        ).otherwise(F.lit(None))
    )
)

# Seleciona as colunas necessárias
df_silver_financeiro = (
    df_silver_financeiro
    .withColumnRenamed("id", "id_filme")
    .select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_pct")

)

df_silver_financeiro.write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

In [0]:
# Métricas
df_silver_metricas = spark.table(f"{bronze_schema}.tb_movies_metrics")

# Normaliza a popularidade para o padrão de um número decimal
def limpar_numero_decimal(coluna):
    # Remove símbolos e letras
    t = F.regexp_replace(F.trim(F.col(coluna)), r"[^0-9.,\-]", "")

    # Retira as vírgulas e os pontos, mantendo o último ponto como separador decimal
    ultimo_virgula = F.length(F.regexp_extract(t, r"^.*,", 0)) > F.length(F.regexp_extract(t, r"^.*\.", 0))
    tem_ambos = t.contains(".") & t.contains(",")

    return (
        F.when(tem_ambos & ultimo_virgula, F.regexp_replace(F.regexp_replace(t, r"\.", ""), ",", "."))
         .when(tem_ambos & ~ultimo_virgula, F.regexp_replace(t, ",", ""))
         # Só vírgula: é decimal ("12,5" -> "12.5").
         .when(t.contains(",") & ~t.contains("."), F.regexp_replace(t, ",", "."))
         .otherwise(t)
    )

# Converte as strings para double/int
df_silver_metricas = (
    df_silver_metricas
    .withColumn("popularidade", limpar_numero_decimal("popularity").try_cast("double"))
    .withColumn("nota_media_tmdb", F.regexp_replace(F.trim("vote_average"), ",", ".").try_cast("double"))
    .withColumn("nota_media_imdb", F.regexp_replace(F.trim("averageRating"), ",", ".").try_cast("double"))
    .withColumn("qtd_votos_tmdb", F.trim("vote_count").try_cast("int"))
    .withColumn("qtd_votos_imdb", F.trim("numVotes").try_cast("int"))
    .withColumn("id_filme", F.trim("id"))
)

def nota_valida(coluna):
    return F.when((F.col(coluna) >= 0) & (F.col(coluna) <= 10), F.col(coluna))

# Contagens e popularidade negativas são inválidas.
def nao_negativo(coluna):
    return F.when(F.col(coluna) >= 0, F.col(coluna))

df_silver_metricas = (
    df_silver_metricas
    # Valida as notas, popularidade e quantidade
    .withColumn("nota_media_tmdb", nota_valida("nota_media_tmdb"))
    .withColumn("nota_media_imdb", nota_valida("nota_media_imdb"))
    .withColumn("qtd_votos_tmdb", nao_negativo("qtd_votos_tmdb"))
    .withColumn("qtd_votos_imdb", nao_negativo("qtd_votos_imdb"))
    .withColumn("popularidade", nao_negativo("popularidade"))
)

# Deduplicação
w = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_silver_metricas = (
    df_silver_metricas
    .filter(F.col("id_filme").isNotNull() & (F.col("id_filme") != ""))
    .withColumn("row_number", F.row_number().over(w))
    .filter(F.col("row_number") == 1)

    # Seleciona as colunas necessárias
    .select(
        "id_filme", "popularidade",
        "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb",
    )
)

df_silver_metricas.write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

In [0]:
# Avaliações dos usuários
df_bronze_avaliacoes = spark.table(f"{bronze_schema}.tb_movies_reviews")

# Renomeia as colunas e converte a nota para double
df_silver_avaliacoes = (
    df_bronze_avaliacoes
    .select(
        F.trim(F.col("id")).alias("id_filme"),
        F.trim(F.col("nome")).alias("nome_usuario"),
        F.regexp_replace(F.trim(F.col("nota")), ',', '.').try_cast("double").alias("nota_usuario"),
        F.col("comentario").cast("string").alias("comentario_usuario"),
        F.col("ingestion_datetime"),
    )
)

# Checa se a nota está no intervalo permitido
df_silver_avaliacoes = df_silver_avaliacoes.withColumn(
    "nota_usuario",
    F.when((F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10), F.col("nota_usuario"))
)

df_silver_avaliacoes = (
    df_silver_avaliacoes
    # Comentários null, vazios ou só com espaço se tornam "sem comentário"
    .withColumn("comentario_usuario", F.when(
        F.col("comentario_usuario").isNull() | (F.trim(F.col("comentario_usuario")) == ""), "Sem comentário"
    ).otherwise(F.trim(F.col("comentario_usuario"))))
    # Retira duplicatas
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
    # Seleciona só as colunas necessárias
    .select("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario")
)

df_silver_avaliacoes.write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")


In [0]:
# Gêneros
df_bronze_credits_and_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_silver_generos = (
    df_bronze_credits_and_tags
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    # Substitui outros separadores (; ou |) por ,
    .withColumn("genres", F.regexp_replace(F.col("genres"), r"[;|]", ","))
    # Explode a coluna de gênero
    .withColumn("genero", F.explode(F.split(F.col("genres"), ",")))
    .withColumn("genero", F.trim(F.col("genero")))
    # Remove resíduos
    .filter(F.col("genero") != "") # Tira gêneros em branco
    .filter(F.col("genero").rlike(r"^[A-Za-zÀ-ÿ][A-Za-zÀ-ÿ\s\-&']*$") )# Tira gêneros com números e símbolos
    .filter(F.length("genero") <= 30) # Tira textos muito longos
    # Deduplicação e seleção só das colunas de id e gênero
    .filter(F.col("id_filme").isNotNull() & (F.col("id_filme") != ""))
    .select("id_filme", "genero")
    .dropDuplicates(["id_filme", "genero"])
)

df_silver_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_schema}.tb_generos")

In [0]:
# Pessoas empresas

# Recebe a coluna da tabela (cast, directors, etc.) e seu nome em português (ator, diretor, etc.)
# e retorna um df com todas as entidades daquele tipo (ex: todos os atores)
def explodir(coluna, entidade):
    return (
        df_bronze_credits_and_tags
        # Renomeia o id
        .withColumn("id_filme", F.trim(F.col("id")))
        # Substitui os separadores que podem ser diferente de vírgula
        .withColumn("entidade", F.regexp_replace(F.col(coluna), r"[;|]", ","))
        # Explode a coluna
        .withColumn("nome_entidade", F.explode(F.split(F.col("entidade"), ",")))
        .withColumn("nome_entidade", F.trim(F.col("nome_entidade")))
        # Coloca o tipo de entidade
        .withColumn("tipo_entidade", F.lit(entidade))
        .select("id_filme", "nome_entidade", "tipo_entidade")
    )

# Explode cada uma das entidades
df_ator = explodir("cast", "Ator")
df_diretor = explodir("directors", "Diretor")
df_roteirista = explodir("writers", "Roteirista")
df_produtora = explodir("production_companies", "Produtora")

# Junta todas as tabelas em uma só
df_silver_pessoas_empresas = (
    df_ator
    .union(df_diretor)
    .union(df_roteirista)
    .union(df_produtora)
)

df_silver_pessoas_empresas = (
    # Descarta entidades com nomes inválidos
    df_silver_pessoas_empresas
    .filter(F.col("nome_entidade") != "") # Tira nomes vazios
    .filter(F.col("nome_entidade").rlike(r"[A-Za-zÀ-ÿ]")) # Tira nomes sem letras
    .filter(F.length("nome_entidade") <= 80) # Tira nomes muito longos
    # Padroniza a capitalização
    .withColumn("nome_entidade", F.initcap(F.col("nome_entidade")))
    # Deduplicação e id nulo
    .filter(F.col("id_filme").isNotNull() & (F.col("id_filme") != ""))
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

df_silver_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")